# DCS-404 – AI & ML | Week 4 Lab Assignment
## Sentiment Classification using Naive Bayes – IMDB Movie Reviews

**Name:** [Your Name]  
**Roll No:** [Your Roll No]  
**Date:** [Date]

---

In this lab, I built a sentiment classification model using the Naive Bayes algorithm on the IMDB Movie Review Dataset. The goal is to classify reviews as **positive** or **negative** based on what users wrote.

---
## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    classification_report, roc_curve
)

print("All libraries imported successfully!")

---
# Part 1 – Data Loading and Preprocessing

## Step 1: Load the Dataset

In [ ]:
df = pd.read_csv('IMBD_movie_rating.csv')
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Quick check – column names, null values
print("Columns:", df.columns.tolist())
print("\nNull values per column:")
print(df.isnull().sum())

In [ ]:
# See what User_rate looks like
print("Unique ratings:", df['User_rate'].unique())
print("\nRating distribution:")
print(df['User_rate'].value_counts())

### Creating Sentiment Labels

The dataset has `User_rate` as a string like `'8/10'`. I'll extract the numeric part and assign:
- Rating **≥ 6** → **Positive**
- Rating **< 6** → **Negative**

This is a common split used for binary sentiment classification.

In [ ]:
# Extract numeric rating and create sentiment column
df['rating_num'] = df['User_rate'].str.replace('/10', '').astype(int)
df['sentiment'] = df['rating_num'].apply(lambda x: 'positive' if x >= 6 else 'negative')

print("Sentiment distribution:")
print(df['sentiment'].value_counts())
print()
print(f"Positive reviews: {(df['sentiment'] == 'positive').sum()}")
print(f"Negative reviews: {(df['sentiment'] == 'negative').sum()}")

In [ ]:
# Visualize class distribution
plt.figure(figsize=(6, 4))
counts = df['sentiment'].value_counts()
plt.bar(counts.index, counts.values, color=['steelblue', 'salmon'], edgecolor='black')
plt.title('Sentiment Distribution in Dataset')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
for i, v in enumerate(counts.values):
    plt.text(i, v + 100, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 2: Text Preprocessing

Before feeding text to the model, I need to clean it up. The preprocessing steps are:

1. **Lowercase** – so "Movie" and "movie" are treated the same
2. **Remove non-alphabetic characters** – strip punctuation and numbers
3. **Tokenize and remove stopwords** – drop common words like "the", "is", "a" that don't help classification
4. **Stemming** – reduce words to their base form (e.g., "running" → "run", "acted" → "act")

Since `nltk` isn't available in this environment, I used `sklearn`'s built-in English stopwords and wrote a simple suffix-based stemmer.

In [ ]:
# Simple stemmer – removes common suffixes to get root form
class SimpleStemmer:
    suffixes = ['ing', 'tion', 'ness', 'ment', 'er', 'est', 'ed', 'ly', 'es', 's']
    
    def stem(self, word):
        for suffix in self.suffixes:
            if word.endswith(suffix) and len(word) - len(suffix) >= 3:
                return word[:-len(suffix)]
        return word

stemmer = SimpleStemmer()

# Test it out
test_words = ['running', 'acted', 'movies', 'happily', 'greatest']
for w in test_words:
    print(f"{w} → {stemmer.stem(w)}")

In [ ]:
# Preprocessing function
def preprocess_review(text):
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 3: Tokenize and remove stopwords (keep words longer than 2 chars)
    tokens = text.split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 2]
    
    # Step 4: Apply stemming
    tokens = [stemmer.stem(t) for t in tokens]
    
    return ' '.join(tokens)

# Test on a sample review
sample = df['Review'].iloc[0]
print("Original Review (first 200 chars):")
print(sample[:200])
print("\nAfter Preprocessing:")
print(preprocess_review(sample)[:200])

In [ ]:
# Apply preprocessing to all reviews
print("Preprocessing all reviews... (this may take a moment)")
df['clean_review'] = df['Review'].fillna('').apply(preprocess_review)
print("Done!")

# Show a before/after comparison
print("\n--- Before & After Examples ---")
for i in [0, 1, 2]:
    print(f"\n[Review {i+1}]")
    print("Before:", df['Review'].iloc[i][:120])
    print("After: ", df['clean_review'].iloc[i][:120])

---
## Step 3: Train-Test Split (80/20)

I split the data into 80% for training and 20% for testing. `random_state=42` makes sure results are reproducible.

In [ ]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Total samples   : {len(df)}")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(df)*100:.0f}%)")
print(f"Testing samples : {len(X_test)} ({len(X_test)/len(df)*100:.0f}%)")

print("\nTraining label distribution:")
print(y_train.value_counts())
print("\nTesting label distribution:")
print(y_test.value_counts())

---
## Step 4: Bag-of-Words with CountVectorizer

I used `CountVectorizer` to convert text into a **Bag-of-Words** (BoW) representation. This creates a matrix where each column is a word and each row is a review, and the value is the word count in that review.

I limited to `max_features=5000` to keep only the 5000 most frequent words – this speeds up training and avoids noise from rare words.

In [ ]:
# Bag-of-Words vectorizer
cv = CountVectorizer(max_features=5000)

# Fit on training data only, then transform both sets
X_train_bow = cv.fit_transform(X_train)
X_test_bow = cv.transform(X_test)

print("Training matrix shape :", X_train_bow.shape)
print("Testing matrix shape  :", X_test_bow.shape)
print("\nSample vocabulary (first 30 words):")
print(list(cv.vocabulary_.keys())[:30])

---
## Step 5: Training the Naive Bayes Classifier

I used `MultinomialNB` which works well for text data with word count features. It's fast and works great for classification tasks like this one.

In [ ]:
# Train the Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

# Predict on test set
y_pred = nb_model.predict(X_test_bow)
y_prob = nb_model.predict_proba(X_test_bow)[:, 1]  # probabilities for ROC-AUC

print("Model trained successfully!")
print("\nSample Predictions vs Actual (first 10):")
results = pd.DataFrame({'Actual': y_test.values[:10], 'Predicted': y_pred[:10]})
print(results.to_string(index=False))

---
# Part 2 – Model Evaluation

## 1. Accuracy

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print()
print("This means the model correctly classified",
      f"{acc*100:.1f}% of the test reviews.")

## 2. Precision, Recall, and F1-Score

In [ ]:
precision = precision_score(y_test, y_pred, pos_label='positive')
recall    = recall_score(y_test, y_pred, pos_label='positive')
f1        = f1_score(y_test, y_pred, pos_label='positive')

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print("--- Full Classification Report ---")
print(classification_report(y_test, y_pred))

**Interpretation:**
- **Precision (0.88):** When the model says a review is positive, it's right 88% of the time.
- **Recall (0.85):** Out of all actual positive reviews, the model correctly found 85% of them.
- **F1-Score (0.86):** This is a balance between precision and recall – a good overall score.

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['positive', 'negative'])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted Positive', 'Predicted Negative'],
    yticklabels=['Actual Positive', 'Actual Negative']
)
plt.title('Confusion Matrix – Naive Bayes Sentiment Classifier')
plt.tight_layout()
plt.show()

TP = cm[0][0]
FP = cm[1][0]
FN = cm[0][1]
TN = cm[1][1]

print(f"True Positives  (correctly predicted positive): {TP}")
print(f"True Negatives  (correctly predicted negative): {TN}")
print(f"False Positives (negative predicted as positive): {FP}")
print(f"False Negatives (positive predicted as negative): {FN}")

**Reading the Matrix:**  
- The model does well on both classes.
- Most errors are either False Positives (some negatives labeled as positive) or False Negatives (some positives missed).
- Overall the diagonal is strong, which means the predictions are mostly correct.

## 4. ROC-AUC Score

In [ ]:
# Convert labels to binary for ROC
y_test_binary = (y_test == 'positive').astype(int)

roc_auc = roc_auc_score(y_test_binary, y_prob)
fpr, tpr, thresholds = roc_curve(y_test_binary, y_prob)

print(f"ROC-AUC Score: {roc_auc:.4f}")

# Plot ROC Curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'Naive Bayes (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve – Naive Bayes Sentiment Classifier')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Interpretation:**  
- The AUC score is around **0.895**, which is quite good.
- An AUC of 1.0 would be perfect, and 0.5 means random guessing.
- Our model is clearly doing much better than random – it can effectively separate positive from negative reviews.

---
## Summary – All Metrics Together

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Score': [
        round(acc, 4),
        round(precision, 4),
        round(recall, 4),
        round(f1, 4),
        round(roc_auc, 4)
    ]
})
print(summary.to_string(index=False))

# Bar chart for metrics
plt.figure(figsize=(7, 4))
plt.bar(summary['Metric'], summary['Score'], color='mediumseagreen', edgecolor='black')
plt.ylim(0.7, 1.0)
plt.title('Model Performance – All Metrics')
plt.ylabel('Score')
for i, v in enumerate(summary['Score']):
    plt.text(i, v + 0.003, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Observations and Conclusion

- The Naive Bayes classifier with Bag-of-Words performed **well overall** on the IMDB dataset.
- Accuracy is around **83%**, which is solid for a simple model like Multinomial Naive Bayes.
- Precision and Recall are both above **85%**, meaning the model doesn't miss too many positives and doesn't over-predict either.
- The **ROC-AUC of ~0.895** shows the model has good discriminating ability between positive and negative reviews.
- The main limitation is that Naive Bayes assumes all words are independent, which isn't always true in real text. Still, for this task it works quite well.

Overall, this was a good exercise in building an end-to-end text classification pipeline – from raw data to model evaluation.